# RSNA Knee Abnormality Detection — `Knee MRI: twelve findings from a single model` 解説付き写し

- **コンペ**: [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)（Research / Code Competition, 賞金 $77,000, 2,790チーム, 残り約2ヶ月）
- **原著者**: DREAD DEVELOPMENT
- **元notebook**: https://www.kaggle.com/code/dreaddevelopment/knee-mri-twelve-findings-from-a-single-model
- **Public Score / Best Score**: **0.924 / 0.924**
- **実行時間**: 53秒（GPU T4 x2） / 102 votes, 7 comments

> ⚠️ これは**学習目的の解説付き写し**です。原著者のコードは変更しておらず、出力は含んでいません。
> 元のnotebookはKaggleの**スクリプト形式カーネル**（1本の `.py`）だったため、原文を論理的な区切り（`# ====` のセクション見出し）で6セルに分割し、各セルの前に日本語の解説セルを追加しています。**コードの中身は1文字も変更していません。**

---

## なぜこのnotebookを選んだか

このコンペではこれまで20本近くのnotebookを扱ってきましたが、その大半が**「複数モデルのランクアンサンブル」**でした。上位帯（0.936）はほぼ全て、他人の重みやDINOバックボーンを組み合わせたブレンドです。

このnotebookは違います。**単一モデル、アンサンブルなし、TTAなし**で0.924を出しています。そして著者は「3本アンサンブルにしても実際のLBでは 0.914 → 0.914〜0.915 にしかならなかった。オフラインの45症例ゴールドセットで見えた +0.010 は**ゴールドセットのノイズだった**」と、コード中のコメントで明記しています。

**アンサンブルの効果を実測して、割に合わないと判断して捨てている**——この判断過程が読める点で、本日の3本の中で最も学習価値が高いと判断しました。

---

## このコンペは何をするのか

膝のMRI検査（1件の study）から、**12種類の所見**を同時に判定します:

ACL断裂 / MCL断裂 / 内側半月板断裂 / 外側半月板断裂 / 内側・外側・膝蓋大腿の変形性関節症（OA） / 関節液貯留（effusion） / 滑膜炎（synovitis） / ベーカー嚢腫 / 骨挫傷 / 骨折

これは**マルチラベル分類**です（1件が複数の所見を同時に持ちうる。多クラス分類とは違う）。

## 評価指標（metric）

**macro-AUC**（12所見それぞれの ROC AUC を計算し、単純平均したもの）。

- **ROC AUC**: 「ランダムな陽性例のスコアが、ランダムな陰性例のスコアより高い確率」。順位のみに依存し、しきい値に依存しない。
- **macro（マクロ平均）**: **12所見を等しい重みで平均**する。所見ごとの症例数は考慮しない。

**なぜこの指標か**: 骨折やベーカー嚢腫のような**稀な所見**は、症例数で重み付けする micro 平均だと事実上無視されます。しかし臨床的には稀な所見の見落としこそ問題です。macro平均にすることで、**「頻出所見だけ当てて稀な所見を捨てる」戦略に高得点を与えない**設計になっています。

## このnotebookの手法が指標をどう最適化しているか

macro-AUCが「12所見を等しく扱う」ことを、著者はアーキテクチャに直接反映させています。

1. **所見ごとに独立したattention重み**（後述の `RaptorClassifier`）。ACL断裂は矢状断の数スライスにしか写らず、変形性関節症は冠状断の多数スライスに広がる。**単一のプーリングだとこの2つが「どのスライスが重要か」を共有させられてしまう**。所見ごとにattentionを分けることで、各所見が自分に必要なスライスだけを見られます。macro平均では稀な所見1つの改善が全体の1/12を動かすので、これは直接スコアに効きます。
2. **ソフトラベルによる学習データの拡張**。構造化ラベルは4,407症例中58件しかない。残りはフリーテキストの読影レポートだけ。著者は**LLMでレポートを読み、12個の確率に変換**して4,349症例の学習データを作りました。「断裂が疑われる」というレポートは 1 ではなく **0.8** になる。曖昧な記述を無理に0/1にしないので、**AUCの順位付けに効く微妙な確信度の差**が保たれます。
3. **周辺スライスを捨てない**。スライスを stack の 6%〜94% から取る（従来は 15%〜85%）。外側半月板と側副靭帯はまさに周辺スライスに存在するため、切り捨てると**その所見だけAUCが落ち、macro平均を直撃**します。
4. **58症例のゴールドセットは学習に一切使わない**。ここで 0.9167 macro-AUC。**正直な評価を確保するために、貴重なラベル付きデータを敢えて使わない**という判断です。


## 【解説 1】notebook冒頭のdocstring — 手法の全体設計

**何をしているか (What)**
実行されるコードはありません。このnotebookの設計全体を説明する長いdocstringです。ただし内容は極めて重要なので、要点を日本語で整理します。

### 学習ラベルをどう作ったか

コンペは4,407症例を提供しますが、**構造化ラベルが付いているのは58症例だけ**。残りはフリーテキストの読影レポートのみ。そのままでは学習できません。

著者は**言語モデルでレポートを読み、12個の確率に変換**しました。「断裂が疑われる」という記述は 1 ではなく **0.8 付近の数値**になります。これで4,349症例の学習データが得られ、**58症例は一度も学習に使わず**、正直な評価用に取っておきました（そこで macro-AUC 0.9167）。

### 形の違う症例から固定サイズの入力を作る

このコンペの本当の難所は「ネットワークの設計」ではなく、**症例ごとに形が全部違う**ことです。1つのstudyは複数のDICOMシリーズを持ち、撮影面（矢状断/冠状断/軸位断）も枚数もバラバラ。固定サイズ入力を要求するネットワークには、何かを決めてやる必要があります。

著者の方式は **5つの固定スロットに合計64枚**を必ず埋める:

| スロット | 撮影面 | fluid-sensitive | 枚数 |
|---|---|---|---|
| 1 | 矢状断 (Sagittal) | 優先する | 18 |
| 2 | 矢状断 | 優先しない | 14 |
| 3 | 冠状断 (Coronal) | 優先する | 12 |
| 4 | 冠状断 | — | 8 |
| 5 | 軸位断 (Axial) | — | 12 |

**fluid-sensitive（水感受性）シーケンスを一部のスロットでだけ優先するのは意図的です。** 水感受性シーケンス（T2/STIRなど）は腫脹・関節液・急性外傷をよく写し、そうでないシーケンスは解剖構造や軟骨をよく写す。12所見はこの両方に分かれて存在するので、**どちらか一方に寄せてはいけない**。

スロットに該当するシリーズが無い場合は**ゼロで埋め、モデルには「ここはスキップしろ」と伝えます**（誤解を招くデータを食わせない）。

### mm単位でクロップする理由

各スライスはDICOMヘッダのピクセル間隔（PixelSpacing）を使い、**中心から140mm四方**で切り出してから336pxにリサイズします。ピクセル数ではなく**ミリメートルで切る**のがポイント。

そうすると、0.3mm/px で撮った検査でも 0.5mm/px で撮った検査でも、**膝がフレームに占める割合が同じ**になります。医学的に無意味なスケール差をモデルに学習させずに済みます。

### 3スライスを3チャンネルに積む

隣接する3枚のスライスを1枚の画像のRGB 3チャンネルに詰めます。これでネットワークは中央スライスの**上下の情報も少しだけ**見られます。**3Dモデルの恩恵の大部分を、2Dモデルのコストで得る**という定番の妥協案です（2.5Dアプローチと呼ばれます）。

*用語*:
- **DICOM** — 医用画像の標準フォーマット。画素値だけでなく、撮影条件・患者情報・ピクセル間隔などのメタデータを大量に含みます。
- **fluid-sensitive sequence（水感受性シーケンス）** — 水（＝浮腫・関節液・炎症）が明るく写るMRI撮像法。T2強調やSTIRなど。
- **ソフトラベル (soft label)** — 0/1ではなく確率値で与える教師信号。曖昧さを情報として保存できます。


In [ ]:
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""


## 【解説 2】設定値 — そして「アンサンブルをやめた理由」がコメントに書いてある

**何をしているか (What)**
importと固定設定です。ここは**コメントの方がコードより重要**です。

主な設定:

| 定数 | 値 | 意味 |
|---|---|---|
| `IMG` | 336 | 1スライスのピクセルサイズ |
| `CROP_MM` | 140.0 | クロップする物理サイズ（mm） |
| `MAXS` | 64 | 1 study あたりのスライス総数 |
| `K_EVAL` | 42 | 推論時に使うウィンドウ数 |
| `LAB` | 12所見のリスト | 出力の順序を固定 |
| `ARMS` | **1要素のみ** | 使うモデル = CoAtNet 1本だけ |

`torch.backends.cudnn.benchmark = True` と `HF_HUB_OFFLINE=1` も設定されています。

**なぜそうするのか (Why)**

**(1) アンサンブルを捨てた理由 —— このnotebookで最も価値のある判断**

コメントによれば、著者は7本のモデルパネルから貪欲前方選択と全部分集合探索の両方で {CoAtNet + Swin + EffNetV2-L} の3本ブレンドを選び、45症例のゴールドセットで **0.9068**（CoAtNet単体 0.9025）を得ました。**+0.004 の改善**です。

ところが**実際のリーダーボードでは**:

> CoAtNet単体 = 0.914、どのブレンドも 0.914〜0.915

つまり **オフラインで見えた改善は再現しなかった**。著者の結論は明確です：

> 「45症例ゴールドセットで見えた ~+0.010 は**ゴールドセットのノイズだった**。1本なら実行時間も1/3で済む。」

**45症例で測った差は、そもそも統計的に意味を持てるサイズではありません。** 貪欲探索と全探索が「一致した」ことは正しさの証拠に見えますが、**両方とも同じノイズの多い45症例を見ている**ので、一致するのは当然です。これは検証設計の失敗であって、探索アルゴリズムの成功ではない。

**(2) 学習データ拡張の効果は正しく測られている**

同じコメントに、58症例のゲートで測った比較があります:

- 旧モデル（3,155症例で学習）: 0.8923
- 本モデル（4,349症例で学習）: **0.9054**（+0.0131、**2000回のブートストラップの92.7%で優位**）

こちらは**ブートストラップで有意性を確認しています**。改善が大きい所見も明示: 外側半月板 +0.071、骨折 +0.057、外側OA +0.048。

**(1)と(2)の対比が教訓です。同じ著者が、片方は「ノイズだった」と捨て、もう片方は「ブートストラップで92.7%」と採用している。差の大きさではなく、差の信頼性で判断している。**

**(3) T4のbf16問題**
コメントに「T4（Turing世代）のcuDNN v9はfp16/fp32のconvエンジンは持つが、**このshapeに対するbf16エンジンは無い**」とあります。ハードウェア世代ごとの対応精度の違いは、Kaggleの無料GPUを使う上で実際にぶつかる制約です。

*用語*:
- **CoAtNet** — 畳み込み（Conv）と自己注意（Attention）を組み合わせたハイブリッド構造。畳み込みの局所性・データ効率と、attentionの大域的な文脈把握を両取りする狙い。
- **ブートストラップ (bootstrap)** — 手元のデータから復元抽出で疑似サンプルを大量に作り、統計量のばらつきを推定する手法。「この差は偶然か」を答えるのに使えます。
- **貪欲前方選択 (greedy forward selection)** — 空集合から始め、最も改善するモデルを1つずつ追加していくアンサンブル構築法。


In [ ]:
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
IMG = 336
CROP_MM = 140.0
# 64 slices per study instead of 44, same proportions. Must match the corpus the weights
# were trained on (knee_corpus_v4.py).
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
         ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(s[2] for s in SLOTS)                     # 64
K_EVAL = 42   # every window position the volume holds, not an evenly spaced subset
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [
    {"file": "raptor_ft_coatnet_v4_full.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]




## 【解説 3】モデル定義 — 所見ごとに独立したattention

**何をしているか (What)**
2つの部品を定義します。

**`build_backbone`**: `timm` からバックボーンを作ります。注目すべきは分岐条件です。

```python
hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", ...))
```

`"coatnet"` や `"maxvit"` は文字列に **"vit" を含みますが、ViTではありません**（conv-attentionハイブリッドで、CLSトークンも補間可能な位置埋め込みも持たない）。だから先に `hybrid` を判定して、ViTの経路に流れないようにしています。ViT系は `global_pool="token"`、それ以外は `global_pool="avg"`。

**`RaptorClassifier`**: 本体です。

```python
self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                         nn.Linear(256, n))    # n = 12
self.clsW = nn.Parameter(torch.zeros(n, F_dim))
```

`head` の処理:
1. K個のウィンドウ特徴を LayerNorm
2. attention 層が **(K ウィンドウ) × (12 所見)** のスコアを出す
3. `softmax(dim=1)` で**ウィンドウ方向に**正規化 → 所見ごとに「どのスライスを見るか」の重み
4. `einsum("bkn,bkf->bnf")` で所見ごとに重み付き和 → 所見ごとに1つの特徴ベクトル
5. 所見ごとの重みベクトル `clsW` との内積 + バイアス → 12個のロジット

**なぜそうするのか (Why)**

**これが本notebookの中核アイデアです。**

普通のマルチラベル分類なら、K個のスライス特徴を平均プーリングして1本のベクトルにし、12出力の線形層に通します。しかしそれだと**12所見が「どのスライスが重要か」という1つの見解を共有せざるを得ません**。

実際には:
- **ACL断裂** — 矢状断の2〜3スライスにしか写らない
- **変形性関節症** — 冠状断の多数スライスに広がって存在する

この2つに同じプーリングを強制するのは無理があります。著者の設計では **attentionの出力が12次元**あり、softmaxがウィンドウ方向にかかるので、**所見ごとに独立した「スライスの重み配分」**が学習されます。

これは **attention-based MIL（Multiple Instance Learning）** の一種で、「症例（bag）の中に複数のスライス（instance）があり、どのinstanceが陽性判定の根拠かは教えられていない」という設定にぴったり当てはまります。

**`load_model` のコメントも重要**: `DataParallel` を**意図的に削除**しています。理由は「隠しテストセット全体を回すと、forwardごとのモジュール複製でシステムRAMがOOMした」。そして main では**アームを逐次実行**するので、ピークRAMは常にモデル1本分。

これは Code Competition 特有の制約です。**手元では動くが、本番の隠しテスト（数倍のサイズ）で落ちる**——スコアが下がるのではなく提出が丸ごと失敗します。実験環境と本番環境の規模差を意識した設計が要ります。

*用語*:
- **timm** — PyTorch Image Modelsライブラリ。数百の事前学習済み画像モデルを統一APIで扱えます。
- **MIL (Multiple Instance Learning)** — 「袋（bag）にラベルが付いているが、袋の中のどの要素がその理由かは不明」という学習設定。病理画像・医用画像で頻出。
- **einsum** — アインシュタインの縮約記法でテンソル積を書く記法。`"bkn,bkf->bnf"` は「k（ウィンドウ）で縮約する」の意。


In [ ]:
# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res




## 【解説 4】推論時のウィンドウ生成と確率計算

**何をしているか (What)**

**`_eval_centers`**: マスク（実際に画像が入っているスライス）から有効範囲 `[lo, hi]` を求め、その内側で中心になれるスライスを列挙し、`np.linspace` で k 個を等間隔に選びます。有効スライスが3枚未満なら先頭3枚にフォールバック。

**`eval_windows`**: 各中心 c について `[c-1, c, c+1]` の3枚を3チャンネルに積み、255で割って[0,1]に、必要ならbilinearで解像度を合わせ、ImageNetの平均/標準偏差で正規化します。

**`infer_probs`**: CUDAなら `torch.autocast(dtype=float16)` で推論。**RuntimeErrorが出たらキャッシュを空にしてfp32で再試行**します。

**`rankpct`**: 列ごとにパーセンタイル順位へ変換（`argsort` を2回かけるイディオム）。

**なぜそうするのか (Why)**

- **fp16→fp32 のフォールバックが重要**: T4でfp16 convが失敗しうることは設定セルのコメントにありました。ここで例外を握って fp32 で再試行するので、**1症例たりとも提出から欠落しません**。Code Competitionでは1件の欠落が提出全体を無効にしかねないので、この保険は「遅くなるが必ず出る」を選んでいます。**速度より完遂を優先する**判断です。
- **`rankpct` の理由**: 評価指標がAUCで**順位にしか依存しない**ので、複数アームをブレンドするなら生の確率ではなく**順位空間で混ぜるべき**です。モデルAが全体的に高い確率を出す癖があっても、順位に直せばその癖は消えます。
- **`K_EVAL = 42`**: コメントに「ボリュームが持つ全てのウィンドウ位置であって、等間隔の部分集合ではない」とあります。64スライスのうち端を除くと使える中心が42個。**全部使う**という選択です。


In [ ]:
# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(device)
    use_cuda = device != "cpu" and str(device).startswith("cuda")
    if use_cuda:
        # fp16 conv on T4 is fully cuDNN-supported (bf16 is NOT -> "no engine").
        try:
            with torch.autocast("cuda", dtype=torch.float16):
                o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
        except RuntimeError:
            # fp32 always has a Turing conv engine; slower but never drops a study.
            torch.cuda.empty_cache()
            o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
    o = torch.sigmoid(model(x).float())
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))




## 【解説 5】前処理 — DICOMから固定形状のボリュームを作る

**何をしているか (What)**
このnotebookで最も泥臭く、最も効く部分です。

**`order_and_meta`**: シリーズ内のDICOMファイルを**正しい順番に並べます**。ファイル名順ではありません。`ImageOrientationPatient`（撮影面の向き）から法線ベクトルを計算し、`ImagePositionPatient`（患者座標系での位置）をその法線に射影した値でソートします。メタデータが欠けている場合のみ `InstanceNumber` にフォールバック。

**`read_px`**: 画素を読み、`apply_modality_lut` で装置固有の値からモダリティ本来の値に変換。さらに **`PhotometricInterpretation == 'MONOCHROME1'` なら白黒を反転**します。

**`mm_crop_resize`**: `CROP_MM / PixelSpacing` でピクセル数を計算し、中心から正方形に切り出して336pxへ。

**`build_study`**: 5スロットを順に埋めます。スロットごとに撮影面とfluid-sensitiveの希望に合うシリーズを選び（既に使ったシリーズは除外）、**stackの6%〜94%から等間隔にk枚**を取り、**そのスロット内の全画素の2〜98パーセンタイル**で輝度を正規化してから uint8 化します。該当シリーズが無ければゼロのまま `idx` を進め、最後に「どのスライスに実データがあるか」の `mask` を返します。

**なぜそうするのか (Why)**

**(1) 3つの「メタデータを読まないと壊れる」罠**

医用画像を扱ったことがないと、この3つは必ず踏みます。

- **スライス順**: DICOMのファイル名は撮影順・位置順を保証しません。法線への射影で並べるのが正しい方法です。間違えると**解剖学的に無意味な順番のボリューム**をモデルに食わせることになります。
- **MONOCHROME1**: DICOMには「値が大きいほど**暗い**」という規約の画像が存在します。これを見落とすと、一部の症例だけ**白黒反転した画像**で学習/推論することになります。エラーは出ません。静かに精度が落ちるだけです。
- **PixelSpacing**: 上述の通り、mm単位で切らないとスケールがバラバラになります。

**(2) 6%〜94%という範囲**

コメントに理由が明記されています。「**側副靭帯と外側半月板は、旧来の 0.15〜0.85 のクロップが捨てていた周辺スライスに存在する**」。これは**解剖学の知識が直接ハイパーパラメータになっている**例です。macro-AUCなので、この2所見のAUCが落ちれば全体の1/6が沈みます。

さらに「重みが学習されたコーパス（`knee_corpus_v4.py` の `SPAN_LO`/`SPAN_HI`）と一致していなければならない」という注記もあります。**推論時の前処理が学習時とズレていると、モデルは見たことのない分布の入力を受け取る**ので、これも静かな失敗の典型です。

**(3) パーセンタイル正規化をスロット単位で行う**

`np.percentile(allpx, [2.0, 98.0])` を**スロット内の全スライスまとめて**計算しています。スライスごとに正規化すると、**スライス間の相対的な明るさの情報が消えます**。関節液貯留のように「ここが周りより明るい」ことが所見である場合、それは致命的です。一方でスロット単位にすることで、シーケンス間（T1とT2で輝度分布が全く違う）の差は吸収されます。**どのレベルで正規化するかは、何を情報として残したいかで決まります。**

*用語*:
- **Modality LUT** — DICOMの生画素値を、物理的に意味のある値（CTならHounsfield単位など）へ変換する対応表。
- **パーセンタイルクリッピング** — 上下数%を外れ値として切り捨ててから正規化する手法。少数の極端に明るい画素に引きずられるのを防ぎます。


In [ ]:
# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and len(iop) == 6:
                    r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                    n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps); recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (knee_corpus_v2.py, SPAN_LO/SPAN_HI).
        n = len(files); lo, hi = int(n * 0.06), int(n * 0.94) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        arrs = []; pss = []
        for p in picks:
            fp, ps = files[min(p, n - 1)]
            try:
                arrs.append(read_px(fp)); pss.append(ps)
            except Exception:
                arrs.append(None); pss.append(med_ps)
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask




## 【解説 6】メイン処理 — テストデータの探索、逐次推論、提出生成

**何をしているか (What)**

**`find_test_root` / `find_weight_file`**: `/kaggle/input` 以下から `test.csv` と重みファイルを探します。**`find_weight_file` に重要なコメント**があります：

> 「直接のデータセットマウントを先に見る。**競技のDICOMツリーを再帰globしては絶対にいけない**」

`main` の流れ:
1. `test.csv` / `test_series.csv` を読み、study→series の辞書 `SER` を作る
2. **アームを1本ずつ逐次実行**（`ARMS` は今回1本）。1症例ずつ `build_study` → `eval_windows` → `infer_probs`
3. 例外が出た症例は**確率0.5のまま、名前をログに出して先へ進む**（`FALLBACK`）
4. アームが終わったら `del model; gc.collect(); torch.cuda.empty_cache()`
5. 所見ごとに `rankpct` してから重み付き平均。重みは正規化してから使う
6. `sub_cols` の順序チェック、行の同一性チェック、有限性チェックを**assertで**行ってから書き出し

**なぜそうするのか (Why)**

**(1) 「競技のDICOMツリーを再帰globするな」**
このコンペのテストデータは数千症例分のDICOMで、**ファイル数が膨大**です。`glob(..., recursive=True)` をそこに掛けると、ファイルシステムの走査だけで実行時間制限を食い潰します。**Code Competitionでは、I/Oパターンそのものが失敗要因になります。**

**(2) 例外を握って0.5を入れる設計**
1症例で例外が出たときの選択肢は2つ:
- クラッシュさせる → **提出が丸ごと得られない**
- その症例だけ0.5（無情報）にして続行 → **その1症例分だけAUCが下がる**

macro-AUCで数千症例あるなら、1症例が0.5でも影響は微小です。**部分的な失敗を全体の失敗にしない**設計。ただし `print` でstudy IDと例外型を必ず出しているので、**失敗が静かに埋もれることはありません**。「握りつぶす」のではなく「記録して続行する」。

**(3) アームの逐次実行とメモリ解放**
コメント通り、両アームを同時に保持するとシステムRAMがOOMしました。逐次実行すればピークRAMは1本分。代償として `build_study` をアームごとに再実行しますが、「前処理は推論に比べれば安い」という判断です。**同じモデル・同じ結果（0.8893）を、直列化しただけ**と明記されています。

**(4) 重みを正規化してからブレンド**
```python
_w = _w / _w.sum()
```
コメントに理由があります。「アームを削除/追加したときに**スケールが静かに変わることが決してないように**」。今は1本なので `_w = [1.0]` ですが、将来2本に戻したときに合計が2になってしまう事故を構造的に防いでいます。

**(5) 3つのassert**
```python
assert list(sub.columns) == sub_cols, "column order drift"
assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
assert np.isfinite(sub[LAB].values).all()
```
エラーメッセージが "column order drift" / "row identity drift" と、**何が起きたのかを言葉で説明しています**。列の順序がずれた提出も、行が入れ替わった提出も、**形式エラーにはならず、ただスコアがランダムに近くなるだけ**です。これは正解ラベルなしで検出できる失敗なので、検出しています。

---

## このnotebookから持ち帰るもの

1. **アンサンブルは常に効くわけではない。実測して、割に合わなければ捨てる。** 45症例で見えた +0.010 は再現せず、著者はそれを認めて1本に戻した。
2. **改善を主張するなら、その差が偶然でないことを示す。** 学習データ拡張は2000回ブートストラップの92.7%で優位、と定量化されている。
3. **医用画像では、メタデータを読まないと静かに壊れる。** スライス順序、MONOCHROME1、PixelSpacing。
4. **ドメイン知識はハイパーパラメータになる。** 「側副靭帯は周辺スライスにある」→ クロップ範囲を 6〜94% に。
5. **Code Competitionでは、精度以前に「完走すること」が要件。** メモリ、I/Oパターン、症例単位のフォールバック。


In [ ]:
# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower():
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    reader = _make_reader()
    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]

    # SEQUENTIAL ARMS (the OOM fix): only ONE model is resident at a time, so peak system RAM ==
    # one model == the single-arm champion's footprint (which graded fine at 0.879). Holding both
    # arms simultaneously OOM'd system RAM on the full hidden test. Each study is re-preprocessed
    # per arm (build_study is cheap vs inference) and every per-study buffer is freed. Same models,
    # same windowing, same rank-mean blend -> identical 0.8893 result, just serialized.
    for a, arm in enumerate(ARMS):
        wp = find_weight_file(arm["file"])
        model, res = load_model(wp, arm["arch"], arm["res"], dev)
        print(f"[arm {a}] loaded {arm['file']} | res {res} | {time.time()-t0:.0f}s", flush=True)
        for i, sid in enumerate(test_ids):
            try:
                vol, mask = build_study(sid, SER, tsdir, reader)
                xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM)
                arm_probs[a][i] = infer_probs(model, xw, dev)
                del vol, mask, xw
            except Exception as e:
                print(f"  [arm {a}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            if (i + 1) % 100 == 0 or i + 1 == N:
                print(f"  [arm {a}] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
        del model
        gc.collect()
        if str(dev).startswith("cuda"):
            torch.cuda.empty_cache()
        print(f"[arm {a}] done + freed | {time.time()-t0:.0f}s", flush=True)

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()
